In [1]:
# pip install shared_utils

In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
import requests
import fsspec
from shapely import wkt
import re
from calitp_data_analysis.sql import get_engine
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
# Load the stored ACS dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/census_blocks_data.parquet", "rb") as f:
    blocks_ca_acs = gpd.read_parquet(f)

In [5]:
# Load the stored organization, ridership, stop, data from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/ridership_trips_routes_weekday.csv", "rb") as f:
    ridership_trips_routes_weekday = pd.read_csv(f)
    
with fs.open(f"{GCS_FILE_PATH}/ridership_trips_routes_saturday.csv", "rb") as f:
    ridership_trips_routes_saturday = pd.read_csv(f)
    
with fs.open(f"{GCS_FILE_PATH}/ridership_trips_routes_saturday.csv", "rb") as f:
    ridership_trips_routes_sunday = pd.read_csv(f)

In [6]:
# Load job density data from GCS and select required columns
# Open the GCS file using your existing fs object
with fs.open(f"{GCS_FILE_PATH}/job_density_blockwithrac_2023.parquet", "rb") as f:
    jobdata = pd.read_parquet(f)

# Select only the columns you want, including geometry
jobdata = jobdata[['GEOID', 'jobs_tot_work', 'jobs_tot_home' ]]

# Load pois data from GCS and select required columns
with fs.open(f"{GCS_FILE_PATH}/pois_2026.parquet", "rb") as f:
    pois = gpd.read_parquet(f)


## Spatial Analysis: Stop Buffers and Census Tract Intersections

In [7]:
ridership_trips_routes_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21316 entries, 0 to 21315
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         21316 non-null  object 
 1   feed_key                  21315 non-null  object 
 2   stop_id                   21315 non-null  object 
 3   stop_name                 21316 non-null  object 
 4   stop_code                 20594 non-null  object 
 5   n_arrivals                21315 non-null  float64
 6   n_routes                  21315 non-null  float64
 7   pt_geom                   21315 non-null  object 
 8   day_type                  21316 non-null  object 
 9   n_routes_ferry            21315 non-null  float64
 10  n_routes_rail             21315 non-null  float64
 11  n_routes_other            21315 non-null  float64
 12  n_routes_bus              21315 non-null  float64
 13  route_id_list             21315 non-null  object 
 14  averag

In [8]:
# Drop rows with missing pt_geom
ridership_trips_routes_weekday = ridership_trips_routes_weekday[
    ridership_trips_routes_weekday['pt_geom'].notna() & 
    (ridership_trips_routes_weekday['pt_geom'] != 'nan')
].copy()

ridership_trips_routes_saturday = ridership_trips_routes_saturday[
    ridership_trips_routes_saturday['pt_geom'].notna() & 
    (ridership_trips_routes_saturday['pt_geom'] != 'nan')
].copy()

ridership_trips_routes_sunday = ridership_trips_routes_sunday[
    ridership_trips_routes_sunday['pt_geom'].notna() & 
    (ridership_trips_routes_sunday['pt_geom'] != 'nan')
].copy()

In [9]:
# Ensure pt_geom is string type
ridership_trips_routes_weekday['pt_geom'] = ridership_trips_routes_weekday['pt_geom'].astype(str)
ridership_trips_routes_saturday['pt_geom'] = ridership_trips_routes_saturday['pt_geom'].astype(str)
ridership_trips_routes_sunday['pt_geom'] = ridership_trips_routes_sunday['pt_geom'].astype(str)

In [10]:
# Convert pt_geom column from WKT to shapely geometry
ridership_trips_routes_weekday['geometry'] = ridership_trips_routes_weekday['pt_geom'].apply(wkt.loads)
ridership_trips_routes_saturday['geometry'] = ridership_trips_routes_saturday['pt_geom'].apply(wkt.loads)
ridership_trips_routes_sunday['geometry'] = ridership_trips_routes_sunday['pt_geom'].apply(wkt.loads)

# Create a GeoDataFrame
gdf_ridership = gpd.GeoDataFrame(ridership_trips_routes_weekday, geometry='geometry')
gdf_ridership_saturday = gpd.GeoDataFrame(ridership_trips_routes_saturday, geometry='geometry')
gdf_ridership_sunday = gpd.GeoDataFrame(ridership_trips_routes_sunday, geometry='geometry')

In [11]:
# Set CRS (assuming WGS84)
gdf_ridership.set_crs(epsg=4326, inplace=True)
gdf_ridership_saturday.set_crs(epsg=4326, inplace=True)
gdf_ridership_sunday.set_crs(epsg=4326, inplace=True)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date,geometry
0,Samtrans,db97cc02836aa5f0cf647d80160b23ec,345017,1000 El Camino Real-Menlo College,345017,64.0,1.0,POINT(-122.191284 37.457543),Sunday,8.800000,15.600000,2025-08-01,2025-08-31,POINT (-122.19128 37.45754)
1,Golden Gate Transit,de77cb40e92fb47fa8d16228977cfb86,40469,1011 Andersen Dr,40469,4.0,1.0,POINT(-122.504252 37.955391),Sunday,1.500000,0.000000,2025-09-01,2025-09-30,POINT (-122.50425 37.95539)
2,Long Beach Transit,cddd375786d835389a7beb9632369907,355,10th & Long Beach NW,0355,32.0,1.0,POINT(-118.189862 33.779026),Sunday,2.300620,26.405652,2024-07-01,2025-06-30,POINT (-118.18986 33.77903)
3,Long Beach Transit,cddd375786d835389a7beb9632369907,356,10th & Pine NW,0356,64.0,1.0,POINT(-118.192676 33.779068),Sunday,98.589716,76.027869,2024-07-01,2025-06-30,POINT (-118.19268 33.77907)
4,SDMTS,1fff52f9349da228c56eef492df5001b,11656,10th Av & Broadway,11656,120.0,2.0,POINT(-117.15566399 32.7159774),Sunday,46.287992,39.811145,2024-09-01,2025-01-25,POINT (-117.15566 32.71598)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12316,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70091,San Mateo,NaN,33.0,1.0,POINT(-122.323851 37.568087),Sunday,478.543508,NaN,2023-11-01,2025-07-31,POINT (-122.32385 37.56809)
12317,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70241,Santa Clara,NaN,33.0,1.0,POINT(-121.93608 37.353238),Sunday,387.793441,NaN,2023-11-01,2025-07-31,POINT (-121.93608 37.35324)
12318,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70041,South San Francisco,NaN,33.0,1.0,POINT(-122.404979051 37.655941395),Sunday,163.026362,NaN,2023-11-01,2025-07-31,POINT (-122.40498 37.65594)
12319,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70221,Sunnyvale,NaN,33.0,1.0,POINT(-122.031372 37.378916),Sunday,593.758215,NaN,2023-11-01,2025-07-31,POINT (-122.03137 37.37892)


In [12]:
# Reproject to match census tracts CRS
gdf_ridership = gdf_ridership.to_crs(blocks_ca_acs.crs)
gdf_ridership_saturday = gdf_ridership_saturday.to_crs(blocks_ca_acs.crs)
gdf_ridership_sunday = gdf_ridership_sunday.to_crs(blocks_ca_acs.crs)

In [13]:
stop_buffered = gdf_ridership.copy()
stop_buffered_saturday = gdf_ridership_saturday.copy()
stop_buffered_sunday = gdf_ridership_sunday.copy()

stop_buffered["geometry"] = stop_buffered.geometry.buffer(404.672)
stop_buffered_saturday["geometry"] = stop_buffered_saturday.geometry.buffer(404.672)
stop_buffered_sunday["geometry"] = stop_buffered_sunday.geometry.buffer(404.672)

In [14]:
# Inner join with ACS data on 'geo_id'
blocks_ca_acs = blocks_ca_acs.merge(jobdata, on = 'GEOID', how='left')

In [15]:
pois = pois.to_crs(blocks_ca_acs.crs)
pois_with_block = gpd.sjoin(
    pois,
    blocks_ca_acs[["GEOID", "geometry"]],
    how="left",
    predicate="within"
)


In [16]:
poi_total = (
    pois_with_block
    .groupby("GEOID")
    .size()
    .reset_index(name="poi_total")
)

In [17]:
poi_total.head(5)

,GEOID,poi_total
0,060014001001,13
1,060014001002,5
2,060014002001,40
3,060014002002,23
4,060014003001,43


In [18]:
blocks_ca_acs = blocks_ca_acs.merge(poi_total, on="GEOID", how="left")
blocks_ca_acs["poi_total"] = blocks_ca_acs["poi_total"].fillna(0)

In [19]:
blocks_ca_acs.head(5)

,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOIDFQ,GEOID,NAME,NAMELSAD,LSAD,ALAND,AWATER,geometry,state,county,block group,county_name,total_pop,median_household_income,employed_pop,households_no_vehicle,total_youth,inc_extremelylow,inc_verylow,inc_low,inc_total_lowincome,area_m2,jobs_tot_work,jobs_tot_home,poi_total
0,06,073,010601,1,1500000US060730106011,060730106011,1,Block Group 1,BG,739477,3511,"POLYGON ((268816.912 -592917.261, 268817.085 -...",6,73,1,Census Tract 106.01,929,211875,457,0,87,29,14,9,52,7.823089e+05,475.0,373.0,5.0
1,06,079,013000,1,1500000US060790130001,060790130001,1,Block Group 1,BG,724518353,27447720,"MULTIPOLYGON (((-106426.951 -247177.326, -1065...",6,79,1,Census Tract 130,1760,82021,657,0,272,82,70,123,275,7.238044e+08,809.0,708.0,21.0
2,06,001,428600,2,1500000US060014286002,060014286002,2,Block Group 2,BG,594782,1003447,"POLYGON ((-200418.485 -25161.808, -200399.126 ...",6,1,2,Census Tract 4286,1837,138125,1186,82,267,20,47,92,159,5.826665e+05,97.0,984.0,1.0
3,06,059,099417,3,1500000US060590994173,060590994173,3,Block Group 3,BG,1424116,1049996,"MULTIPOLYGON (((179728.252 -476305.429, 179833...",6,59,3,Census Tract 994.17,834,250001,356,0,204,0,0,31,31,1.380024e+06,70.0,531.0,1.0
4,06,085,505800,2,1500000US060855058002,060855058002,2,Block Group 2,BG,642980,0,"POLYGON ((-172553.178 -74536.512, -172430.186 ...",6,85,2,Census Tract 5058,937,209195,492,0,224,20,34,12,66,6.468382e+05,5598.0,517.0,251.0


In [20]:
blocks_ca_acs.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 25610 entries, 0 to 25609
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   STATEFP                  25610 non-null  object  
 1   COUNTYFP                 25610 non-null  object  
 2   TRACTCE                  25610 non-null  object  
 3   BLKGRPCE                 25610 non-null  object  
 4   GEOIDFQ                  25610 non-null  object  
 5   GEOID                    25610 non-null  object  
 6   NAME                     25610 non-null  object  
 7   NAMELSAD                 25610 non-null  object  
 8   LSAD                     25610 non-null  object  
 9   ALAND                    25610 non-null  int64   
 10  AWATER                   25610 non-null  int64   
 11  geometry                 25610 non-null  geometry
 12  state                    25610 non-null  int64   
 13  county                   25610 non-null  int64   
 14

In [21]:
blocks_ca_acs.crs

<Projected CRS: EPSG:3310>
Name: NAD83 / California Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: California Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [22]:
stop_buffered.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 21315 entries, 0 to 21315
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   organization_name         21315 non-null  object  
 1   feed_key                  21315 non-null  object  
 2   stop_id                   21315 non-null  object  
 3   stop_name                 21315 non-null  object  
 4   stop_code                 20594 non-null  object  
 5   n_arrivals                21315 non-null  float64 
 6   n_routes                  21315 non-null  float64 
 7   pt_geom                   21315 non-null  object  
 8   day_type                  21315 non-null  object  
 9   n_routes_ferry            21315 non-null  float64 
 10  n_routes_rail             21315 non-null  float64 
 11  n_routes_other            21315 non-null  float64 
 12  n_routes_bus              21315 non-null  float64 
 13  route_id_list             21315 non-null  o

In [23]:
geometry_intersect = gpd.overlay(
    stop_buffered, 
    blocks_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)


geometry_intersect_saturday = gpd.overlay(
    stop_buffered_saturday, 
    blocks_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)


geometry_intersect_sunday = gpd.overlay(
    stop_buffered_sunday, 
    blocks_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)

In [24]:
# Calculate intersected area
geometry_intersect['area_2'] = geometry_intersect.geometry.area
geometry_intersect_saturday['area_2'] = geometry_intersect_saturday.geometry.area
geometry_intersect_sunday['area_2'] = geometry_intersect_sunday.geometry.area

# Calculate the proportion of the tract that intersects each stop
geometry_intersect['area_ratio'] = geometry_intersect['area_2'] / geometry_intersect['area_m2']
geometry_intersect_saturday['area_ratio'] = geometry_intersect_saturday['area_2'] / geometry_intersect_saturday['area_m2']
geometry_intersect_sunday['area_ratio'] = geometry_intersect_sunday['area_2'] / geometry_intersect_sunday['area_m2']

In [25]:
geometry_intersect.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 99017 entries, 0 to 99016
Data columns (total 49 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   organization_name         99017 non-null  object  
 1   feed_key                  99017 non-null  object  
 2   stop_id                   99017 non-null  object  
 3   stop_name                 99017 non-null  object  
 4   stop_code                 95933 non-null  object  
 5   n_arrivals                99017 non-null  float64 
 6   n_routes                  99017 non-null  float64 
 7   pt_geom                   99017 non-null  object  
 8   day_type                  99017 non-null  object  
 9   n_routes_ferry            99017 non-null  float64 
 10  n_routes_rail             99017 non-null  float64 
 11  n_routes_other            99017 non-null  float64 
 12  n_routes_bus              99017 non-null  float64 
 13  route_id_list             99017 non-nu

In [26]:
geometry_intersect['lowincome_total'] = (
    geometry_intersect['inc_extremelylow'] + 
    geometry_intersect['inc_verylow'] 
)

In [27]:
# Define demographic and socioeconomic columns to be adjusted by area ratio
cols_to_weight = [
    'total_pop', 'median_household_income', 'employed_pop', 'households_no_vehicle', 
    'total_youth', 'lowincome_total',
    'inc_extremelylow', 'inc_verylow', 'inc_low', 'inc_total_lowincome', 'jobs_tot_work', 'jobs_tot_home', 'poi_total'
]

# Apply area_ratio
for col in cols_to_weight:
    geometry_intersect[f'{col}_adj'] = geometry_intersect[col] * geometry_intersect['area_ratio']

for col in cols_to_weight:
    geometry_intersect_saturday[f'{col}_adj'] = geometry_intersect_saturday[col] * geometry_intersect_saturday['area_ratio']

for col in cols_to_weight:
    geometry_intersect_sunday[f'{col}_adj'] = geometry_intersect_sunday[col] * geometry_intersect_sunday['area_ratio']

In [28]:
geometry_intersect.organization_name.unique()

array(['Gold Coast Transit', 'Samtrans', 'SDMTS', 'Fresno County',
       'SacRT Bus', 'San Francisco Bay Area Rapid Transit District',
       'Orange County Transportation Authority', 'Long Beach Transit',
       'Foothill Transit', 'Golden Gate Park Shuttle', 'Big Blue Bus',
       'Culver City Bus', 'Riverside Transit', 'Caltrain',
       'City of Burbank'], dtype=object)

In [29]:
stop_acs_rollup = geometry_intersect.groupby(
    ['feed_key', 'stop_id', 'organization_name'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight]].sum()

stop_acs_rollup_saturday = geometry_intersect_saturday.groupby(
    ['feed_key', 'stop_id', 'organization_name'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight]].sum()

stop_acs_rollup_sunday = geometry_intersect_sunday.groupby(
    ['feed_key', 'stop_id', 'organization_name'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight]].sum()

In [30]:
stop_route_df = gdf_ridership.merge(
    stop_acs_rollup,
    on=['feed_key', 'stop_id','organization_name'],
    how='left'
)


stop_route_df_saturday = gdf_ridership_saturday.merge(
    stop_acs_rollup_saturday,
    on=['feed_key', 'stop_id','organization_name'],
    how='left'
)

stop_route_df_sunday = gdf_ridership_sunday.merge(
    stop_acs_rollup_sunday,
    on=['feed_key', 'stop_id','organization_name'],
    how='left'
)

In [31]:
stop_route_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 21315 entries, 0 to 21314
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   organization_name            21315 non-null  object  
 1   feed_key                     21315 non-null  object  
 2   stop_id                      21315 non-null  object  
 3   stop_name                    21315 non-null  object  
 4   stop_code                    20594 non-null  object  
 5   n_arrivals                   21315 non-null  float64 
 6   n_routes                     21315 non-null  float64 
 7   pt_geom                      21315 non-null  object  
 8   day_type                     21315 non-null  object  
 9   n_routes_ferry               21315 non-null  float64 
 10  n_routes_rail                21315 non-null  float64 
 11  n_routes_other               21315 non-null  float64 
 12  n_routes_bus                 21315 non-null  float64

In [32]:
stop_route_df = gpd.GeoDataFrame(
    stop_route_df, 
    geometry='geometry', 
    crs=geometry_intersect.crs
)


stop_route_df_saturday = gpd.GeoDataFrame(
    stop_route_df_saturday, 
    geometry='geometry', 
    crs=geometry_intersect_saturday.crs
)

stop_route_df_sunday = gpd.GeoDataFrame(
    stop_route_df_sunday, 
    geometry='geometry', 
    crs=geometry_intersect_sunday.crs
)

In [33]:
# Store data in warehouse_block
with fs.open(f"{GCS_FILE_PATH}/stop_route_df_block.parquet", "wb") as f:
    stop_route_df.to_parquet(f, index=False)

with fs.open(f"{GCS_FILE_PATH}/stop_route_df_saturday_block.parquet", "wb") as f:
    stop_route_df_saturday.to_parquet(f, index=False)

with fs.open(f"{GCS_FILE_PATH}/stop_route_df_sunday_block.parquet", "wb") as f:
    stop_route_df_sunday.to_parquet(f, index=False)